# Partial ozone column, rankings, and event profiles

Raw inputs are read from `PAPER1_ARCHIVE_ROOT`; preprocessing products are read from `PAPER1_PREPROCESSED_ROOT`; new diagnostics are written to `PAPER1_DERIVED_ROOT` (default: repository-local `work/`). Source files are never modified.


## Roots and write policy

Declares the read-only public input tree and isolated staging tree. No public product is overwritten.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "analysis",
        Path.cwd() / "Paper1" / "analysis",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/analysis/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARCH_HINDCAST_ROOT = Path(os.environ.get(
    "PAPER1_MARCH_HINDCAST_SOURCE",
    ""
    "",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only March hindcast root:", MARCH_HINDCAST_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)


## WACCM 207 + 23 spring manifest and one master low-25 threshold

Computes exact-interface 30--70 hPa partial-column ozone, the centered five-day minimum over exactly 1 March--30 April, and one 230-event rank. The 57th-lowest value is the only WACCM low-25 threshold; the accepted server values are regression assertions, not algorithm inputs.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
from paper1_diagnostics import (
    RANKING_COLUMNS, parse_year, rank_events, spring_minimum,
    waccm_partial_ozone,
)

def build_waccm_segment(segment, root, expected_events):
    arrays, rows = [], []
    paths = sorted((root / "O3").glob("*.O3.nc"))
    if not paths:
        raise FileNotFoundError(f"No WACCM O3 files under {root / 'O3'}")
    for path in paths:
        model_year = parse_year(path)
        with xr.open_dataset(path, decode_times=True, chunks={"time": 16}) as source:
            partial = waccm_partial_ozone(source).load()
        partial = partial.assign_coords(model_year=("time", np.full(partial.sizes["time"], model_year)))
        arrays.append(partial)
        try:
            rows.append(spring_minimum(
                partial, source_family="WACCM", source_segment=segment,
                event_year=model_year, model_year=model_year,
                target=(segment == "BWCN" and model_year == 8),
            ))
        except ValueError as error:
            print("excluded incomplete spring:", path.name, error)
    if len(rows) != expected_events:
        raise RuntimeError(f"{segment}: {len(rows)} complete springs; expected {expected_events}")
    combined = xr.concat(arrays, dim="time").sortby(["model_year", "time"]).to_dataset()
    combined.attrs.update(
        product_version=PRODUCT_VERSION,
        method="exact 30--70 hPa interface overlap; 60--90N cosine mean",
        source_segment=segment,
        complete_spring_count=expected_events,
    )
    target = product_path("ozone", f"{segment.lower()}_partial_o3.nc")
    write_netcdf_atomic(
        combined, target,
        required_vars={"partial_o3_du": ("time",)},
        required_coords=("date", "model_year"), overwrite=OVERWRITE,
    )
    return rows

long_rows = build_waccm_segment(
    "LONGRUN", PREPROCESSED_ROOT / "B2000WCN001002_timefixed", 207
)
bwcn_rows = build_waccm_segment("BWCN", PREPROCESSED_ROOT / "BWCN", 23)
if (len(long_rows), len(bwcn_rows)) != (207, 23):
    raise RuntimeError("WACCM manifest must be exactly 207 LONGRUN + 23 BWCN")

waccm_master = rank_events(long_rows + bwcn_rows, expected_count=230)
if int(waccm_master.is_low25.sum()) != 57:
    raise RuntimeError("floor(0.25 * 230) must select exactly 57 WACCM events")
threshold = float(waccm_master.low25_threshold_du.iloc[0])
target_minimum = float(waccm_master.loc[waccm_master.is_target, "minimum_du"].iloc[0])
bwcn_minimum_id = (
    waccm_master.loc[waccm_master.source_segment == "BWCN"]
    .sort_values(["minimum_du", "event_id"], kind="mergesort")
    .iloc[0].event_id
)
if bwcn_minimum_id != "BWCN:0008":
    raise RuntimeError(f"Restart reference must be the lowest BWCN spring; found {bwcn_minimum_id}")
if not np.isclose(threshold, 96.83588104248048, atol=1.0e-3):
    raise RuntimeError(f"Server regression failed for WACCM master threshold: {threshold}")
if not np.isclose(target_minimum, 73.43604278564453, atol=1.0e-3):
    raise RuntimeError(f"Server regression failed for BWCN year-0008 minimum: {target_minimum}")

write_csv_atomic(
    waccm_master, product_path("ozone", "waccm_master_rankings.csv"),
    required_columns=RANKING_COLUMNS, exact_rows=230, overwrite=OVERWRITE,
)
write_csv_atomic(
    waccm_master.sort_values(["source_segment", "model_year"]),
    product_path("ozone", "waccm_event_manifest.csv"),
    required_columns=RANKING_COLUMNS, exact_rows=230, overwrite=OVERWRITE,
)
print("WACCM master threshold (DU):", threshold)


## MERRA-2 1980--2025 ranking

Computes the same ozone metric after kg/kg-to-mol/mol conversion. All 46 springs are required and ranked independently (11 low-25 events).

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
from paper1_diagnostics import merra2_partial_ozone, rank_events, spring_minimum

paths = [PREPROCESSED_ROOT / "MERRA2_Processed" / "O3" / f"MERRA2.O3.{year}.nc" for year in range(1980, 2026)]
missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"MERRA-2 1980--2025 is incomplete: {missing[:3]}")

pieces, rows = [], []
for year, path in zip(range(1980, 2026), paths):
    with xr.open_dataset(path, decode_times=True, chunks={"time": 16}) as source:
        partial = merra2_partial_ozone(source).load()
    pieces.append(partial)
    rows.append(spring_minimum(
        partial, source_family="MERRA2", source_segment="MERRA2",
        event_year=year, model_year=year, target=(year == 2020),
    ))
merra = xr.concat(pieces, dim="time").sortby("time").to_dataset()
merra.attrs.update(
    product_version=PRODUCT_VERSION, source_years="1980--2025",
    method="MERRA-2 kg/kg to mol/mol; exact-boundary 30--70 hPa; 60--90N cosine mean",
)
write_netcdf_atomic(
    merra, product_path("ozone", "merra2_1980_2025_partial_o3.nc"),
    required_vars={"partial_o3_du": ("time",)}, required_coords=("date",),
    overwrite=OVERWRITE,
)
merra_rankings = rank_events(rows, expected_count=46)
if int(merra_rankings.is_low25.sum()) != 11:
    raise RuntimeError("MERRA-2 low25 must be ranked independently: floor(46/4)=11")
write_csv_atomic(
    merra_rankings, product_path("ozone", "merra2_rankings.csv"),
    required_columns=RANKING_COLUMNS, exact_rows=46, overwrite=OVERWRITE,
)


## Three 30-member restart ozone products

January and February use CAM hybrid interfaces; March uses March pressure-level molar mixing ratio. All retain `member`, `time`, and integer `date` coordinates.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
from paper1_diagnostics import parse_member, pressure_level_partial_ozone, waccm_partial_ozone

def write_members(case, root, pressure_level=False):
    if pressure_level:
        candidates = sorted(root.glob("*.nc"))
    else:
        candidates = sorted((root / "O3").glob("*.O3.nc"))
    if len(candidates) != 30:
        raise RuntimeError(f"{case}: expected 30 O3 members; found {len(candidates)}")
    arrays, labels, reference_date = [], [], None
    for path in candidates:
        with xr.open_dataset(path, decode_times=True, chunks={"time": 16}) as source:
            partial = (
                pressure_level_partial_ozone(source, mass_mixing_ratio=False)
                if pressure_level else waccm_partial_ozone(source)
            ).load()
        dates = np.asarray(partial.date.values, dtype=np.int32)
        if reference_date is None:
            reference_date = dates
        elif not np.array_equal(reference_date, dates):
            raise RuntimeError(f"{case}: member calendars differ")
        arrays.append(partial.drop_vars("date"))
        labels.append(parse_member(path))
    combined = xr.concat(arrays, dim=xr.IndexVariable("member", labels)).to_dataset()
    combined = combined.assign_coords(date=("time", reference_date))
    combined.attrs.update(
        product_version=PRODUCT_VERSION, case=case, member_count=30,
        method="exact 30--70 hPa boundaries; centered five-day minimum deferred to verification",
    )
    write_netcdf_atomic(
        combined, product_path("ozone", f"hindcast_{case}_partial_o3.nc"),
        required_vars={"partial_o3_du": ("member", "time")},
        required_coords=("member", "date"), exact_sizes={"member": 30},
        overwrite=OVERWRITE,
    )

write_members("0008-01", PREPROCESSED_ROOT / "Hindcast" / "0008-01")
write_members("0008-02", PREPROCESSED_ROOT / "Hindcast" / "0008-02")
write_members("0008-03", MARCH_HINDCAST_ROOT, pressure_level=True)


## Appendix B event-profile bootstrap

Input: daily January–May 60–90° N ozone profiles. MERRA-2 2020 is compared with the fixed 1980–2019 baseline (N=40); WACCM year 0008 is compared with the other 22 complete BWCN years. The target is excluded from both baseline pools. No temporal smoothing is applied to the event anomaly, climatology, or bootstrap significance test. The 5,000-resample 95% percentile interval is evaluated independently at every day and pressure level.

Output: `ozone/event_profile_bootstrap5000.nc`, including the daily anomaly, target-excluded climatology, confidence bounds, significance mask, explicit baseline IDs/counts, and `temporal_smoothing_days=0` metadata.


In [ ]:
from paper1_diagnostics import (
    PLEV_HPA, bootstrap_event_profile, cosine_latitude_mean, date_int,
    hybrid_mid_pressure, log_pressure_interpolate, noleap_index, parse_year,
    select_latitude, _require_ozone_units,
)

PROFILE_LEVELS = PLEV_HPA[(PLEV_HPA >= 1.0) & (PLEV_HPA <= 100.0)]
OCTOBER_START = 273

def polar_profile(path, *, merra=False):
    with xr.open_dataset(path, decode_times=True, chunks={"time": 12}) as ds:
        if merra:
            _require_ozone_units(ds["O3"], mass_mixing_ratio=True)
            field = ds["O3"] * (28.9647 / 47.9982)
            pressure_name = "lev" if "lev" in field.coords else "plev"
            field = field.interp({pressure_name: PROFILE_LEVELS})
            if pressure_name != "plev":
                field = field.rename({pressure_name: "plev"})
        else:
            _require_ozone_units(ds["O3"], mass_mixing_ratio=False)
            field = log_pressure_interpolate(ds["O3"], hybrid_mid_pressure(ds), PROFILE_LEVELS)
        field = select_latitude(field, 60.0, 90.0)
        if "lon" in field.dims:
            field = field.mean("lon", skipna=True)
        field = cosine_latitude_mean(field, 60.0, 90.0).transpose("time", "plev").load()
        dates = date_int(ds)
    return field.assign_coords(date=("time", dates))

def january_may_profile(profile, event_year):
    """Return one complete, no-leap January--May daily profile (151 days)."""
    dates = np.asarray(profile.date.values, dtype=np.int64)
    years = dates // 10000
    months = (dates % 10000) // 100
    days = dates % 100
    keep = (years == event_year) & (months >= 1) & (months <= 5)
    keep &= ~((months == 2) & (days == 29))
    selected = profile.isel(time=np.flatnonzero(keep)).sortby("date")
    if selected.sizes["time"] != 151:
        raise ValueError(
            f"event {event_year}: expected 151 complete no-leap Jan--May days"
        )
    return np.asarray(selected.values, dtype=float)

# MERRA-2: fixed 1980--2019 baseline and 2020 event, matching Appendix B.
merra_labels = list(range(1980, 2021))
merra_events = []
for year in merra_labels:
    profile = polar_profile(
        PREPROCESSED_ROOT / "MERRA2_Processed" / "O3" / f"MERRA2.O3.{year}.nc",
        merra=True,
    )
    merra_events.append(january_may_profile(profile, year))
merra_target = merra_events[merra_labels.index(2020)]
merra_target_id = "MERRA2:2020"
merra_baseline_ids = [f"MERRA2:{year:04d}" for year in range(1980, 2020)]
merra_pool = np.stack(
    [value for year, value in zip(merra_labels, merra_events) if year < 2020]
)
if merra_pool.shape[0] != 40 or merra_target_id in merra_baseline_ids:
    raise RuntimeError("Appendix B requires 40 target-excluded MERRA-2 baseline years")
merra_climatology = np.nanmean(merra_pool, axis=0).astype(np.float32)
merra_anom, merra_lo, merra_hi, merra_sig = bootstrap_event_profile(
    merra_pool, merra_target, repetitions=5000, seed=15001,
)

# WACCM: year 0008 against the other 22 complete BWCN years only.
manifest = pd.read_csv(product_path("ozone", "waccm_event_manifest.csv"))
bwcn_manifest = manifest.loc[manifest.source_segment.eq("BWCN")].copy()
if len(bwcn_manifest) != 23:
    raise RuntimeError("Appendix B requires the 23 complete BWCN event years")
profile_cache = {}
for source_path in sorted((PREPROCESSED_ROOT / "BWCN" / "O3").glob("*.O3.nc")):
    year = parse_year(source_path)
    if year in set(bwcn_manifest.model_year.astype(int)):
        profile_cache[year] = polar_profile(source_path)
waccm_events, waccm_ids = [], []
for row in bwcn_manifest.itertuples(index=False):
    year = int(row.model_year)
    if year not in profile_cache:
        raise FileNotFoundError(f"Missing complete BWCN O3 profile for year {year:04d}")
    waccm_events.append(january_may_profile(profile_cache[year], year))
    waccm_ids.append(row.event_id)
target_id = "BWCN:0008"
if waccm_ids.count(target_id) != 1:
    raise RuntimeError("BWCN:0008 must appear exactly once in the Appendix B pool")
waccm_target = waccm_events[waccm_ids.index(target_id)]
waccm_baseline_ids = [key for key in waccm_ids if key != target_id]
waccm_pool = np.stack(
    [value for key, value in zip(waccm_ids, waccm_events) if key != target_id]
)
if waccm_pool.shape[0] != 22 or target_id in waccm_baseline_ids:
    raise RuntimeError("Appendix B requires 22 target-excluded BWCN baseline years")
waccm_climatology = np.nanmean(waccm_pool, axis=0).astype(np.float32)
waccm_anom, waccm_lo, waccm_hi, waccm_sig = bootstrap_event_profile(
    waccm_pool, waccm_target, repetitions=5000, seed=15002,
)

profile_output = xr.Dataset(
    {
        "merra2_o3_anomaly": (("season_day", "merra2_pressure_hpa"), merra_anom),
        "merra2_o3_climatology": (("season_day", "merra2_pressure_hpa"), merra_climatology),
        "merra2_ci_low": (("season_day", "merra2_pressure_hpa"), merra_lo),
        "merra2_ci_high": (("season_day", "merra2_pressure_hpa"), merra_hi),
        "merra2_significant": (("season_day", "merra2_pressure_hpa"), merra_sig.astype(np.int8)),
        "waccm_o3_anomaly": (("season_day", "waccm_pressure_hpa"), waccm_anom),
        "waccm_o3_climatology": (("season_day", "waccm_pressure_hpa"), waccm_climatology),
        "waccm_ci_low": (("season_day", "waccm_pressure_hpa"), waccm_lo),
        "waccm_ci_high": (("season_day", "waccm_pressure_hpa"), waccm_hi),
        "waccm_significant": (("season_day", "waccm_pressure_hpa"), waccm_sig.astype(np.int8)),
    },
    coords={
        "season_day": np.arange(92, 243),
        "merra2_pressure_hpa": PROFILE_LEVELS,
        "waccm_pressure_hpa": PROFILE_LEVELS,
    },
    attrs={
        "product_version": PRODUCT_VERSION,
        "method": "target-event anomaly relative to target-excluded baseline pool",
        "target_excluded": "True", "bootstrap_replicates": 5000,
        "percentile_bounds": "2.5,97.5", "event_calendar": "Jan--May no-leap",
        "merra2_target_id": merra_target_id,
        "merra2_baseline_event_count": int(merra_pool.shape[0]),
        "merra2_baseline_event_ids": ",".join(merra_baseline_ids),
        "waccm_target_id": target_id,
        "waccm_baseline_event_count": int(waccm_pool.shape[0]),
        "waccm_baseline_event_ids": ",".join(waccm_baseline_ids),
        "ozone_source_units": "MERRA2 kg/kg; WACCM mol/mol",
        "event_profile_baseline": "MERRA2 1980-2019 (N=40); BWCN all complete years except 0008 (N=22)",
        "profile_anomaly_temporal_resolution": "daily",
        "temporal_smoothing_days": 0,
    },
)
profile_output.merra2_pressure_hpa.attrs["units"] = "hPa"
profile_output.waccm_pressure_hpa.attrs["units"] = "hPa"
write_netcdf_atomic(
    profile_output, product_path("ozone", "event_profile_bootstrap5000.nc"),
    required_vars={
        "merra2_o3_anomaly": ("season_day", "merra2_pressure_hpa"),
        "merra2_o3_climatology": ("season_day", "merra2_pressure_hpa"),
        "merra2_ci_low": ("season_day", "merra2_pressure_hpa"),
        "merra2_ci_high": ("season_day", "merra2_pressure_hpa"),
        "merra2_significant": ("season_day", "merra2_pressure_hpa"),
        "waccm_o3_anomaly": ("season_day", "waccm_pressure_hpa"),
        "waccm_o3_climatology": ("season_day", "waccm_pressure_hpa"),
        "waccm_ci_low": ("season_day", "waccm_pressure_hpa"),
        "waccm_ci_high": ("season_day", "waccm_pressure_hpa"),
        "waccm_significant": ("season_day", "waccm_pressure_hpa"),
    },
    exact_sizes={"season_day": 151},
    required_attrs={
        "bootstrap_replicates": 5000, "percentile_bounds": "2.5,97.5",
        "target_excluded": "True", "merra2_target_id": "MERRA2:2020",
        "waccm_target_id": "BWCN:0008",
        "profile_anomaly_temporal_resolution": "daily",
        "temporal_smoothing_days": 0,
        "merra2_baseline_event_count": int(merra_pool.shape[0]),
        "waccm_baseline_event_count": int(waccm_pool.shape[0]),
    },
    overwrite=OVERWRITE,
)
